# M-TTA experiments (notebook edition)
Consolidated from the validated pilot code (`mtta.py` + `mtta_v2.py`, v3 mechanisms).
- Section III of the manuscript (Eqs. 5–22) + reviewer-driven protocol: unsupervised τ, point-wise metrics without PA,
  update precision / anomaly leak rate, cross-entity shift, ablation switches.
- v2/v3 mechanisms: adaptive τ (running quantile, gate-filtered buffer), multi-mask uncertainty gate, u-score fusion diagnostics.

**Usage**: run cells top-to-bottom once (setup → data prep → definitions), then run any campaign cell. Every run writes one JSON;
re-running a campaign cell skips finished runs, so you can stop/restart freely.

In [ ]:
# --- 0. environment -----------------------------------------------------------------------------
# %pip install torch numpy pandas scikit-learn tqdm matplotlib   # uncomment on first use
import os, io, json, time, glob, ast, math, zipfile, collections, subprocess, urllib.request
from dataclasses import dataclass, asdict, field
import numpy as np, pandas as pd, torch, torch.nn as nn
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda": torch.backends.cuda.matmul.allow_tf32 = True
print("device:", DEVICE, "| torch", torch.__version__, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
ROOT = "."                       # project root (data/, results_*/ will be created here)
os.makedirs(f"{ROOT}/data", exist_ok=True)

In [ ]:
# --- 1. configuration ----------------------------------------------------------------------------
@dataclass
class Config:
    dataset: str = "smd"          # "smd" (txt files) or "npy" (<entity>_train.npy / _test.npy / _test_label.npy)
    data_root: str = "data/SMD"
    train_entity: str = "machine-1-1"
    test_entity: str = ""         # set to a different entity for cross-entity (natural shift) evaluation
    tag: str = "M-TTA"
    # model / offline training (Sec. III-B, III-C)
    L: int = 100; hidden: int = 256; latent: int = 128; p_m: float = 0.15
    epochs: int = 50; batch: int = 64; lr: float = 1e-3; train_stride: int = 1
    mask: int = 1; revin: int = 1
    # online stage (Sec. III-D)  — NOTE: Adam is required; plain SGD as written in Eq.(21) is inert
    tta: int = 1; lr_online: float = 1e-3; online_opt: str = "adam"
    # threshold / gate (v2-v3)
    val_frac: float = 0.1; tau_quantile: float = 0.99
    tau_mode: str = "adaptive"    # "fixed" | "adaptive"
    tau_buffer: str = "gated"     # "all" | "gated"   (only used when tau_mode == "adaptive")
    buf_size: int = 5000
    gate: str = "unc"             # "none" | "unc"
    views: int = 8; unc_quantile: float = 0.95
    fuse_lams: tuple = (0.5, 1.0, 2.0)
    # preprocessing
    global_norm: int = 0          # per-channel z-score with TRAINING statistics (use 1 for SWaT / heterogeneous sensors)
    drop_head: int = 0            # rows to drop from the start of training (SWaT: 21600 = 6 h warm-up)
    segments: int = 10; seed: int = 0

# variant presets (Table VI-style ablations + v3 mechanisms)
VARIANTS = {
    "fixed":        dict(mask=1, tta=1, tau_mode="fixed",    gate="none"),                                   # manuscript method (Adam online)
    "noMask_fixed": dict(mask=0, tta=1, tau_mode="fixed",    gate="none"),                                   # Base + RevIN + TTA
    "gate_gbuf5k":  dict(mask=1, tta=1, tau_mode="adaptive", gate="unc", tau_buffer="gated", buf_size=5000), # v3 full mechanism
    "gate_noTTA":   dict(mask=1, tta=0, tau_mode="fixed",    gate="unc", tau_buffer="gated", buf_size=5000), # u-score without adaptation
    "base_noTTA":   dict(mask=0, tta=0, tau_mode="fixed",    gate="none"),                                   # Base + RevIN
    "mask_noTTA":   dict(mask=1, tta=0, tau_mode="fixed",    gate="none"),                                   # Base + Masking + RevIN
}

In [ ]:
# --- 2. data preparation (run once) --------------------------------------------------------------
def prep_smd(root=f"{ROOT}/data/SMD"):
    """Download all 28 SMD machines (train/test/test_label) from the OmniAnomaly repository."""
    os.makedirs(root, exist_ok=True)
    base = "https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/master/ServerMachineDataset"
    machines = [f"machine-1-{i}" for i in range(1, 9)] + [f"machine-2-{i}" for i in range(1, 10)] + [f"machine-3-{i}" for i in range(1, 12)]
    for m in tqdm(machines, desc="SMD"):
        for sub in ("train", "test", "test_label"):
            fn = f"{root}/{sub}_{m}.txt"
            if not os.path.exists(fn):
                urllib.request.urlretrieve(f"{base}/{sub}/{m}.txt", fn)
    return machines

def prep_msl(zip_path, root=f"{ROOT}/data/MSL"):
    """Extract all MSL entities from the NASA telemanom archive (data/data/train|test/<id>.npy + labeled_anomalies.csv)."""
    os.makedirs(root, exist_ok=True)
    z = zipfile.ZipFile(zip_path)
    la = pd.read_csv(io.BytesIO(z.read("labeled_anomalies.csv")))
    ids = []
    for _, r in la[la.spacecraft == "MSL"].iterrows():
        cid = r.chan_id
        trn = np.load(io.BytesIO(z.read(f"data/data/train/{cid}.npy"))).astype(np.float32)
        tst = np.load(io.BytesIO(z.read(f"data/data/test/{cid}.npy"))).astype(np.float32)
        lab = np.zeros(len(tst), np.int64)
        for a, b in ast.literal_eval(r.anomaly_sequences): lab[a:b + 1] = 1
        np.save(f"{root}/{cid}_train.npy", trn); np.save(f"{root}/{cid}_test.npy", tst); np.save(f"{root}/{cid}_test_label.npy", lab)
        ids.append(cid)
    print("MSL entities:", len(ids)); return ids

def prep_swat_official(normal_csv, attack_csv, root=f"{ROOT}/data/SWaT51"):
    """Official iTrust files: SWaT_Dataset_Normal_v1.csv (1 stage-header row to skip) and SWaT_Dataset_Attack_v0.csv."""
    os.makedirs(root, exist_ok=True)
    def read(path, skip):
        df = pd.read_csv(path, skiprows=skip, dtype=str, low_memory=False)
        df.columns = [c.strip() for c in df.columns]
        X = df.iloc[:, 1:52].apply(lambda s: pd.to_numeric(s.str.strip(), errors="coerce")).to_numpy(np.float32)
        lab = (df["Normal/Attack"].str.strip().str.lower() == "attack").to_numpy(np.int64) if "Normal/Attack" in df.columns else None
        return df.columns[1:52].tolist(), X, lab
    ctr, Xtr, _ = read(normal_csv, 1); cte, Xte, lab = read(attack_csv, 0)
    assert ctr == cte and not np.isnan(Xtr).any() and not np.isnan(Xte).any()
    np.save(f"{root}/SWaT_train.npy", Xtr); np.save(f"{root}/SWaT_test.npy", Xte); np.save(f"{root}/SWaT_test_label.npy", lab)
    open(f"{root}/columns.txt", "w").write("\n".join(ctr))
    print("SWaT train", Xtr.shape, "test", Xte.shape, "anomaly ratio", round(float(lab.mean()), 4))

# SMD_MACHINES = prep_smd()
# MSL_IDS = prep_msl("SML__MSL.zip")
# prep_swat_official("swat/SWaT_Dataset_Normal_v1.csv", "swat/SWaT_Dataset_Attack_v0.csv")

In [ ]:
# --- 3. loaders & windowing ------------------------------------------------------------------------
def load_smd(root, machine):
    tr = np.loadtxt(f"{root}/train_{machine}.txt", delimiter=",", dtype=np.float32)
    te = np.loadtxt(f"{root}/test_{machine}.txt", delimiter=",", dtype=np.float32)
    lb = np.loadtxt(f"{root}/test_label_{machine}.txt", dtype=np.int64)
    return tr, te, lb

def load_npy(root, name):
    tr = np.load(f"{root}/{name}_train.npy").astype(np.float32)
    te = np.load(f"{root}/{name}_test.npy").astype(np.float32)
    lb = np.load(f"{root}/{name}_test_label.npy").astype(np.int64).reshape(-1)
    return tr, te, lb

def make_windows(x, L, stride):
    """x: (T, C) CPU tensor -> (N, L, C) strided VIEW (no copy)."""
    T, C = x.shape; n = (T - L) // stride + 1
    return x.as_strided(size=(n, L, C), stride=(stride * C, C, 1))

In [ ]:
# --- 4. model (Sec. III-B/C): mask-aware RevIN, mask token, FC encoder-decoder ---------------------
class MaskAwareRevIN(nn.Module):
    def __init__(self, C, eps=1e-5):
        super().__init__(); self.eps = eps
        self.gamma = nn.Parameter(torch.ones(C)); self.beta = nn.Parameter(torch.zeros(C))
    def stats(self, x, m):                                   # Eq. (7)-(8): observed values only
        n = m.sum(1, keepdim=True)
        mu = (m * x).sum(1, keepdim=True) / (n + self.eps)
        var = (m * (x - mu) ** 2).sum(1, keepdim=True) / (n + self.eps)
        return mu, torch.sqrt(var + self.eps)
    def normalize(self, x, mu, sigma):   return self.gamma * ((x - mu) / sigma) + self.beta          # Eq. (9)
    def denormalize(self, xb, mu, sigma): return sigma * ((xb - self.beta) / self.gamma) + mu        # Eq. (12)

class MTTA(nn.Module):
    def __init__(self, L, C, hidden=256, latent=128, use_revin=True, use_mask_token=True):
        super().__init__()
        self.L, self.C, self.use_revin, self.use_mask_token = L, C, use_revin, use_mask_token
        self.revin = MaskAwareRevIN(C) if use_revin else None
        self.omega = nn.Parameter(torch.zeros(C)) if use_mask_token else None                 # mask token omega_c
        D = L * C
        self.enc1, self.enc2 = nn.Linear(D, hidden), nn.Linear(hidden, latent)
        self.dec1, self.dec2 = nn.Linear(latent, hidden), nn.Linear(hidden, D)                 # dec2 = final decoder layer (online-updated)
        self.act = nn.GELU()
    def final_layer_params(self): return list(self.dec2.parameters())
    def forward(self, x, m=None):
        B = x.shape[0]
        if m is None: m = torch.ones_like(x)
        if self.use_revin:
            mu, sigma = self.revin.stats(x, m); xb = self.revin.normalize(x, mu, sigma)
        else: xb = x
        xt = m * xb + (1 - m) * self.omega if self.use_mask_token else m * xb                # Eq. (10)
        h = self.act(self.enc1(xt.reshape(B, -1))); z = self.act(self.enc2(h)); h = self.act(self.dec1(z))
        xb_hat = self.dec2(h).reshape(B, self.L, self.C)                                       # Eq. (11)
        return self.revin.denormalize(xb_hat, mu, sigma) if self.use_revin else xb_hat

def offline_loss(x, x_hat, m, p_m, eps=1e-8):                                                  # Eqs. (13)-(15)
    se = (x - x_hat) ** 2
    l_obs = (m * se).sum() / (m.sum() + eps); l_mask = ((1 - m) * se).sum() / ((1 - m).sum() + eps)
    return (1 - p_m) * l_obs + p_m * l_mask

def online_loss(x, x_hat, q, eps=1e-8):                                                        # Eq. (20)
    se = ((x - x_hat) ** 2).sum(dim=(1, 2)); L, C = x.shape[1], x.shape[2]
    return (q * se).sum() / (q.sum() * L * C + eps)

@torch.no_grad()
def scores(model, x):                                                                          # Eqs. (16)-(18), (22)
    e = (x - model(x, None)) ** 2
    return e.mean(2), e.mean(2).mean(1), e.mean(1)                                             # a_{i,t}, s_i, r_{i,c}

In [ ]:
# --- 5. offline training, uncertainty, calibration (GPU-resident: no per-batch host<->device traffic) -----
def _gen(seed):
    g = torch.Generator(device=DEVICE); g.manual_seed(seed); return g

def train_offline(model, x_train, cfg, log=None):
    x_train = x_train.to(DEVICE)                                   # whole series on device; windows are strided views
    g = _gen(cfg.seed)
    W = make_windows(x_train, cfg.L, cfg.train_stride); N = W.shape[0]
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr); model.train()
    for ep in range(cfg.epochs):
        perm = torch.randperm(N, generator=g, device=DEVICE); tot = torch.zeros((), device=DEVICE)
        for i in range(0, N, cfg.batch):
            xb = W[perm[i:i + cfg.batch]]
            m = (torch.rand(xb.shape, generator=g, device=DEVICE) >= cfg.p_m).float() if cfg.mask else torch.ones_like(xb)
            loss = offline_loss(xb, model(xb, m), m, cfg.p_m if cfg.mask else 0.0)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); tot += loss.detach() * xb.shape[0]
        if log and ((ep + 1) % max(1, cfg.epochs // 5) == 0): log(f"  epoch {ep+1}/{cfg.epochs} loss {tot.item()/N:.5f}")
    model.eval(); return model

@torch.no_grad()
def mask_view_uncertainty(model, x, p_m, views, gen):
    """u_i = std over V masked views of the mean reconstruction error on MASKED cells. All on device."""
    errs = []
    for _ in range(views):
        m = (torch.rand(x.shape, generator=gen, device=x.device) >= p_m).float()
        se = (x - model(x, m)) ** 2
        errs.append(((1 - m) * se).sum(dim=(1, 2)) / ((1 - m).sum(dim=(1, 2)) + 1e-8))
    E = torch.stack(errs, 0); return E.std(0), E.mean(0)

@torch.no_grad()
def calibrate(model, x_val, cfg, views):
    """tau0 = q-quantile of window scores on the held-out TRAINING slice; u_thr = q_u-quantile of uncertainty. No test labels."""
    gen = _gen(cfg.seed + 1000); x_val = x_val.to(DEVICE)
    W = make_windows(x_val, cfg.L, 1); ss, uu = [], []
    for i in range(0, W.shape[0], 512):
        xb = W[i:i + 512]
        _, s, _ = scores(model, xb); ss.append(s)
        if views > 0: uu.append(mask_view_uncertainty(model, xb, cfg.p_m, views, gen)[0])
    s_all = torch.cat(ss); u_all = torch.cat(uu) if views else None
    u_thr = torch.quantile(u_all, cfg.unc_quantile).item() if views else float("inf")
    return torch.quantile(s_all, cfg.tau_quantile).item(), u_thr, s_all, u_all

In [ ]:
# --- 6. online stage (Sec. III-D + v2/v3) and metrics — GPU-resident, single host sync at the end ---------
def online_test(model, x_test, y_test, cfg, tau0, s_init, u_thr, views, u_init=None):
    x_test = x_test.to(DEVICE); T, C = x_test.shape; L = cfg.L
    W = make_windows(x_test, L, 1); N = W.shape[0]
    Wlab = make_windows(torch.as_tensor(y_test, dtype=torch.float32).unsqueeze(1), L, 1).squeeze(-1)
    win_has_anom = (Wlab.max(1).values > 0).to(DEVICE)                          # EVALUATION ONLY
    gen = _gen(cfg.seed + 2000)
    params = model.final_layer_params()
    opt = (torch.optim.Adam if cfg.online_opt == "adam" else torch.optim.SGD)(params, lr=cfg.lr_online)
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    # running-quantile buffer as a device ring buffer
    K = cfg.buf_size; init = s_init[-K:].to(DEVICE)
    buf = torch.empty(K, device=DEVICE); buf[:len(init)] = init; nbuf = len(init); ptr = nbuf % K
    dev = DEVICE
    point_score = torch.zeros(T, device=dev); point_tau = torch.full((T,), tau0, device=dev); point_u = torch.zeros(T, device=dev)
    win_score = torch.zeros(N, device=dev); win_tau = torch.zeros(N, device=dev); q_all = torch.zeros(N, device=dev); u_all = torch.zeros(N, device=dev)
    n_updates = 0; t0 = time.time()
    for i in range(0, N, cfg.batch):
        xb = W[i:i + cfg.batch]; B = xb.shape[0]
        tau = tau0 if cfg.tau_mode == "fixed" else torch.quantile(buf[:nbuf], cfg.tau_quantile)
        a, s, _ = scores(model, xb)
        win_score[i:i + B] = s; win_tau[i:i + B] = tau
        idx = torch.arange(i, i + B, device=dev) + L - 1
        point_score[idx] = a[:, -1]; point_tau[idx] = tau
        if i == 0: point_score[:L - 1] = a[0, :L - 1]; point_tau[:L - 1] = tau
        q = (s <= tau).float(); u = None
        if views > 0:
            u, _ = mask_view_uncertainty(model, xb, cfg.p_m, views, gen)
            u_all[i:i + B] = u; point_u[idx] = u
            if i == 0: point_u[:L - 1] = u[0]
            if cfg.gate == "unc": q = q * (u <= u_thr).float()
        q_all[i:i + B] = q
        if cfg.tta and bool(q.sum() > 0):
            loss = online_loss(xb, model(xb, None), q); opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); n_updates += 1
        if cfg.tau_mode == "adaptive":
            new = s[u <= u_thr] if (cfg.tau_buffer == "gated" and u is not None) else s
            n_new = new.numel()
            if n_new:
                pos = (ptr + torch.arange(n_new, device=dev)) % K; buf[pos] = new
                ptr = (ptr + n_new) % K; nbuf = min(K, nbuf + n_new)
    if dev.type == "cuda": torch.cuda.synchronize()
    elapsed = time.time() - t0
    for p in model.parameters(): p.requires_grad_(True)
    point_score, point_tau, point_u = point_score.cpu(), point_tau.cpu(), point_u.cpu()
    win_score, win_tau, q_all, win_has_anom = win_score.cpu(), win_tau.cpu(), q_all.cpu(), win_has_anom.cpu()
    # ---- fused-score diagnostics
    fused = {}
    if views > 0 and np.isfinite(u_thr) and u_thr > 0:
        from sklearn.metrics import roc_auc_score, average_precision_score
        lab = np.asarray(y_test)
        if 0 < lab.sum() < len(lab):
            for lam in cfg.fuse_lams:
                f = (point_score / max(tau0, 1e-12) + lam * point_u / u_thr).numpy()
                fused[f"auroc_fused_{lam}"] = float(roc_auc_score(lab, f)); fused[f"auprc_fused_{lam}"] = float(average_precision_score(lab, f))
                if u_init is not None:                                   # unsupervised tau for the FUSED score (validation-slice quantile)
                    f_val = (s_init.detach().cpu() / max(tau0, 1e-12) + lam * u_init.detach().cpu() / u_thr)
                    tau_f = torch.quantile(f_val, cfg.tau_quantile).item()
                    mf = point_metrics(f, lab, tau_f)
                    fused[f"tau_fused_{lam}"] = tau_f; fused[f"f1_unsup_tau_fused_{lam}"] = mf["f1_unsup_tau"]
                    fused[f"precision_unsup_tau_fused_{lam}"] = mf["precision_unsup_tau"]; fused[f"recall_unsup_tau_fused_{lam}"] = mf["recall_unsup_tau"]
            fused["auroc_u_only"] = float(roc_auc_score(lab, point_u.numpy())); fused["auprc_u_only"] = float(average_precision_score(lab, point_u.numpy()))
    sel = q_all.bool()
    st = {"n_windows": int(N), "n_selected": int(sel.sum()), "n_updates": n_updates,
          "n_anomalous_windows": int(win_has_anom.sum()), "n_anomalous_selected": int((sel & win_has_anom).sum()),
          "update_precision": float((sel & ~win_has_anom).sum() / max(1, sel.sum())),
          "anomaly_leak_rate": float((sel & win_has_anom).sum() / max(1, win_has_anom.sum())),
          "normal_kept_rate": float((sel & ~win_has_anom).sum() / max(1, (~win_has_anom).sum())),
          "ms_per_window": 1000.0 * elapsed / N, "segments": [], **fused}
    for k, ix in enumerate(np.array_split(np.arange(N), cfg.segments)):
        it = torch.as_tensor(ix); sk, lk, qk, tk = win_score[it], win_has_anom[it], sel[it], win_tau[it]; normal = ~lk
        st["segments"].append({"segment": k + 1, "fpr": float(((sk > tk) & normal).sum() / max(1, normal.sum())),
                               "update_precision": float((qk & normal).sum() / max(1, qk.sum())),
                               "n_anomalous_selected": int((qk & lk).sum()), "tau": float(tk.mean())})
    return point_score.numpy(), point_tau.numpy(), point_u.numpy(), st

def point_metrics(score, label, tau_arr):
    """Point-wise metrics WITHOUT point adjustment. F1@tau uses the (possibly time-varying) unsupervised threshold."""
    from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
    if not (0 < label.sum() < len(label)): return {k: float("nan") for k in ("auroc", "auprc", "f1_unsup_tau", "f1_best")}
    pred = (score > tau_arr).astype(int)
    tp = int(((pred == 1) & (label == 1)).sum()); fp = int(((pred == 1) & (label == 0)).sum()); fn = int(((pred == 0) & (label == 1)).sum())
    prec, rec = tp / max(1, tp + fp), tp / max(1, tp + fn)
    p, r, _ = precision_recall_curve(label, score)
    return {"auroc": float(roc_auc_score(label, score)), "auprc": float(average_precision_score(label, score)),
            "f1_unsup_tau": 2 * prec * rec / max(1e-12, prec + rec), "precision_unsup_tau": prec, "recall_unsup_tau": rec,
            "f1_best": float(np.max(2 * p * r / np.maximum(p + r, 1e-12)))}

In [ ]:
# --- 7. one experiment --------------------------------------------------------------------------------
def run_experiment(cfg: Config, log=print, return_scores=False):
    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
    loader = load_smd if cfg.dataset == "smd" else load_npy
    tr, te, lb = loader(cfg.data_root, cfg.train_entity)
    if cfg.test_entity and cfg.test_entity != cfg.train_entity: _, te, lb = loader(cfg.data_root, cfg.test_entity)
    if cfg.drop_head > 0: tr = tr[cfg.drop_head:]
    if cfg.global_norm:
        mu_g, sd_g = tr.mean(0, keepdims=True), tr.std(0, keepdims=True); sd_g = np.where(sd_g < 1e-6, 1.0, sd_g)
        tr = ((tr - mu_g) / sd_g).astype(np.float32); te = ((te - mu_g) / sd_g).astype(np.float32)
    C = tr.shape[1]; n_val = max(int(len(tr) * cfg.val_frac), cfg.L + 20)
    if len(tr) - n_val >= cfg.L + 20: x_tr, x_val = torch.as_tensor(tr[:-n_val]), torch.as_tensor(tr[-n_val:])
    else: x_tr, x_val = torch.as_tensor(tr), torch.as_tensor(tr[-(cfg.L + 20):])      # very short entity
    x_te = torch.as_tensor(te)
    model = MTTA(cfg.L, C, cfg.hidden, cfg.latent, bool(cfg.revin), bool(cfg.mask)).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    t0 = time.time(); train_offline(model, x_tr, cfg, log=None); train_time = time.time() - t0
    views = cfg.views if ((cfg.gate == "unc" or cfg.tau_buffer == "gated") and cfg.mask) else 0
    tau0, u_thr, s_val, u_val = calibrate(model, x_val, cfg, views)
    score, tau_arr, u_arr, st = online_test(model, x_te, lb, cfg, tau0, s_val, u_thr, views, u_init=u_val)
    met = point_metrics(score, lb, tau_arr)
    res = {**asdict(cfg), "test_entity": cfg.test_entity or cfg.train_entity, "C": C, "params_M": n_params / 1e6,
           "views_used": views, "tau0": tau0, "u_thr": u_thr, "train_time_s": train_time, **met,
           **{k: v for k, v in st.items() if k != "segments"}, "segments": st["segments"]}
    res["fuse_lams"] = list(cfg.fuse_lams)
    if log: log(f"{cfg.tag:16s} {cfg.train_entity}->{res['test_entity']}  F1@tau={met['f1_unsup_tau']:.3f} AUROC={met['auroc']:.3f} "
                f"AUPRC={met['auprc']:.3f} leak={st['anomaly_leak_rate']:.3f} prec={st['update_precision']:.3f}"
                + (f" | u-AUROC={st['auroc_u_only']:.3f} fused2={st.get('auroc_fused_2.0', float('nan')):.3f}" if views else ""))
    return (res, score, u_arr, lb) if return_scores else res

def run_campaign(cfgs, out_dir, worker=None):
    """Run a list of Configs; skip runs whose JSON already exists (resumable).
    worker=(k, n): this kernel handles every n-th config starting at k -> run n notebook copies in parallel on one GPU."""
    worker = worker or WORKER
    if worker: cfgs = cfgs[worker[0]::worker[1]]
    os.makedirs(out_dir, exist_ok=True); done = 0
    for cfg in tqdm(cfgs, desc=f"{out_dir} w{worker[0]}/{worker[1]}" if worker else out_dir):
        fn = f"{out_dir}/{cfg.tag}__{cfg.train_entity}__{cfg.test_entity or cfg.train_entity}__s{cfg.seed}.json"
        if os.path.exists(fn): done += 1; continue
        try:
            r = run_experiment(cfg, log=tqdm.write); json.dump(r, open(fn, "w")); done += 1
        except Exception as e:
            tqdm.write(f"FAILED {cfg.tag} {cfg.train_entity}: {e!r}")
    print(f"{done}/{len(cfgs)} runs available in {out_dir}")

def make_cfgs(dataset, data_root, entities, variants, seeds, **overrides):
    return [Config(dataset=dataset, data_root=data_root, train_entity=e, tag=v, seed=s, **VARIANTS[v], **overrides)
            for s in seeds for e in entities for v in variants]

## Campaigns
Edit `SEEDS`, `EPOCHS`, `STRIDE` for the full run (manuscript settings: epochs=50, stride=1, 10 seeds). Pilot used 30 / 10 / 1.

In [ ]:
SEEDS = list(range(10)); EPOCHS = 50; STRIDE = 1
WORKER = None          # single kernel. For 3 parallel kernels: copy this notebook 3x and set (0,3), (1,3), (2,3)
SMD_MACHINES = [f"machine-1-{i}" for i in range(1, 9)] + [f"machine-2-{i}" for i in range(1, 10)] + [f"machine-3-{i}" for i in range(1, 12)]
MSL_IDS = ["P-15","M-6","M-1","M-2","S-2","P-10","T-4","T-5","F-7","M-3","M-4","M-5","C-1","C-2","T-12","T-13","F-4","F-5","D-14","T-9","P-14","T-8","P-11","D-15","D-16","M-7","F-8"]
CORE = ["fixed", "noMask_fixed", "gate_gbuf5k", "gate_noTTA", "base_noTTA", "mask_noTTA"]

# quick smoke test (1 machine, 3 epochs)
_ = run_experiment(Config(train_entity="machine-1-1", epochs=3, train_stride=10, tag="smoke", **VARIANTS["gate_gbuf5k"]))

In [ ]:
# --- SMD: 28 machines x variants x seeds
run_campaign(make_cfgs("smd", f"{ROOT}/data/SMD", SMD_MACHINES, CORE, SEEDS, epochs=EPOCHS, train_stride=STRIDE), f"{ROOT}/results_gpu/SMD")

In [ ]:
# --- MSL: 27 entities x variants x seeds
run_campaign(make_cfgs("npy", f"{ROOT}/data/MSL", MSL_IDS, CORE, SEEDS, epochs=EPOCHS, train_stride=STRIDE), f"{ROOT}/results_gpu/MSL")

In [ ]:
# --- SWaT (official 51 channels): global z-score + 6h warm-up drop; 3 seeds
run_campaign(make_cfgs("npy", f"{ROOT}/data/SWaT51", ["SWaT"], ["fixed", "noMask_fixed", "gate_gbuf5k", "gate_noTTA"], [0, 1, 2],
                       epochs=EPOCHS, train_stride=STRIDE, global_norm=1, drop_head=21600), f"{ROOT}/results_gpu/SWaT51")

In [ ]:
# --- SMD cross-entity natural shift: adjacent machines within each group, 3 variants x 5 seeds
pairs = []
for g, n in (("1", 8), ("2", 9), ("3", 11)):
    pairs += [(f"machine-{g}-{i}", f"machine-{g}-{i+1}") for i in range(1, n)]
cfgs = [Config(dataset="smd", data_root=f"{ROOT}/data/SMD", train_entity=a, test_entity=b, tag=f"shift_{v}", seed=s,
               epochs=EPOCHS, train_stride=STRIDE, **VARIANTS[v]) for s in range(5) for a, b in pairs for v in ("fixed", "gate_gbuf5k", "noMask_fixed")]
run_campaign(cfgs, f"{ROOT}/results_gpu/SMD_shift")

## Aggregation

In [ ]:
def load_results(out_dir):
    rows = []
    for fn in glob.glob(f"{out_dir}/*.json"):
        r = json.load(open(fn)); segs = r.pop("segments", None)
        if segs:
            r["early_fpr"] = np.mean([s["fpr"] for s in segs[:3]]); r["late_fpr"] = np.mean([s["fpr"] for s in segs[-3:]])
        rows.append(r)
    df = pd.DataFrame(rows); df["shift"] = df["train_entity"] != df["test_entity"]; return df

pd.set_option("display.width", 220); pd.set_option("display.precision", 3)
METRICS = ["f1_unsup_tau", "f1_best", "auroc", "auprc", "update_precision", "anomaly_leak_rate", "normal_kept_rate", "early_fpr", "late_fpr"]

def summarize(out_dir):
    df = load_results(out_dir); same = df[~df["shift"]]
    print(f"=== {out_dir}: {same.train_entity.nunique()} entities, seeds {sorted(df.seed.unique())} ===")
    print(same.groupby("tag")[METRICS].mean().to_string())
    g = same[same.views_used > 0]
    if len(g):
        cols = ["auroc", "auroc_u_only"] + [c for c in g.columns if c.startswith("auroc_fused")] + ["auprc", "auprc_u_only"] + [c for c in g.columns if c.startswith("f1_unsup_tau_fused")]
        print("\n--- score comparison (variants that compute u) ---"); print(g.groupby("tag")[cols].mean().to_string())
        per = g.groupby(["tag", "train_entity"])[["auroc", "auroc_fused_2.0"]].mean().reset_index()
        for t, gg in per.groupby("tag"): print(f"{t}: fused2 > recon on {int((gg['auroc_fused_2.0'] > gg.auroc).sum())}/{len(gg)} entities")
    return df

# df_smd = summarize(f"{ROOT}/results_gpu/SMD"); df_msl = summarize(f"{ROOT}/results_gpu/MSL")

In [ ]:
# --- plots (per-entity recon vs fused AUROC; segment-wise leak) -----------------------------------------
import matplotlib.pyplot as plt
def plot_score_gain(df, tag="gate_gbuf5k", title=""):
    g = df[(df.tag == tag) & (~df["shift"])].groupby("train_entity")[["auroc", "auroc_fused_2.0"]].mean()
    fig, ax = plt.subplots(figsize=(3.5, 3.5)); ax.scatter(g.auroc, g["auroc_fused_2.0"], s=14)
    lim = [0.3, 1.0]; ax.plot(lim, lim, "k--", lw=0.8); ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("AUROC (reconstruction error)"); ax.set_ylabel("AUROC (fused with u, λ=2)"); ax.set_title(title); fig.tight_layout(); return fig

def plot_segments(out_dir, entity, tags=("fixed", "gate_gbuf5k"), key="fpr"):
    fig, ax = plt.subplots(figsize=(3.5, 2.4))
    for t in tags:
        fns = glob.glob(f"{out_dir}/{t}__{entity}__{entity}__s*.json")
        if not fns: continue
        segs = np.array([[s[key] for s in json.load(open(f))["segments"]] for f in fns])
        ax.plot(range(1, segs.shape[1] + 1), segs.mean(0), marker="o", ms=3, label=t)
    ax.set_xlabel("Test segment"); ax.set_ylabel(key); ax.legend(frameon=False); fig.tight_layout(); return fig

# plot_score_gain(df_smd, title="SMD"); plot_segments(f"{ROOT}/results_gpu/SMD", "machine-1-1")